In [2]:
# ─── Import Library ───────────────────────────────────────────────────────
import re
import math
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer

# Inisialisasi stopword dan stemmer
stop_factory = StopWordRemoverFactory()
stopword_set = set(stop_factory.get_stop_words())

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

print('Library berhasil diimport.')

Library berhasil diimport.


In [3]:
# ─── Definisi 4 Dokumen Pendek ────────────────────────────────────────────
documents = {
    "D1": "Pemeliharaan jaringan komputer kampus dilakukan untuk meningkatkan "
          "performa sistem informasi akademik.",
    "D2": "Kurikulum coding dan kecerdasan buatan diterapkan sebagai mata pelajaran "
          "pilihan di sekolah Indonesia.",
    "D3": "Program beasiswa KIP Kuliah dibuka untuk mahasiswa aktif perguruan tinggi "
          "melalui pendaftaran secara online.",
    "D4": "Serangan siber meningkat drastis di Indonesia akibat penggunaan kecerdasan "
          "buatan oleh pelaku kejahatan digital."
}

print("Dataset dokumen:")
for doc_id, teks in documents.items():
    print(f"  {doc_id}: {teks}")

Dataset dokumen:
  D1: Pemeliharaan jaringan komputer kampus dilakukan untuk meningkatkan performa sistem informasi akademik.
  D2: Kurikulum coding dan kecerdasan buatan diterapkan sebagai mata pelajaran pilihan di sekolah Indonesia.
  D3: Program beasiswa KIP Kuliah dibuka untuk mahasiswa aktif perguruan tinggi melalui pendaftaran secara online.
  D4: Serangan siber meningkat drastis di Indonesia akibat penggunaan kecerdasan buatan oleh pelaku kejahatan digital.


In [ ]:
# ─── Fungsi Preprocessing (sama dengan No.1, Sastrawi) ────────────────────
def preprocess(text):
    
    text = text.lower()
    
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = text.split()
    
    tokens = [t for t in tokens if t not in stopword_set]
    
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens


preprocessed = {doc_id: preprocess(teks) for doc_id, teks in documents.items()}

print('Hasil preprocessing:')
for doc_id, tokens in preprocessed.items():
    print(f'  {doc_id}: {tokens}')

Hasil preprocessing:
  D1: ['pelihara', 'jaring', 'komputer', 'kampus', 'laku', 'tingkat', 'performa', 'sistem', 'informasi', 'akademik']
  D2: ['kurikulum', 'coding', 'cerdas', 'buat', 'terap', 'mata', 'ajar', 'pilih', 'sekolah', 'indonesia']
  D3: ['program', 'beasiswa', 'kip', 'kuliah', 'buka', 'mahasiswa', 'aktif', 'guru', 'tinggi', 'lalu', 'daftar', 'online']
  D4: ['serang', 'siber', 'tingkat', 'drastis', 'indonesia', 'akibat', 'guna', 'cerdas', 'buat', 'laku', 'jahat', 'digital']


In [ ]:
# ─── Bangun Vocabulary dan BoW ────────────────────────────────────────────
from collections import Counter

vocab = sorted(set(t for tokens in preprocessed.values() for t in tokens))
doc_ids = list(preprocessed.keys())

print(f'Ukuran vocabulary: {len(vocab)} term')
print(f'Term: {vocab}')
bow_data = {}
for doc_id, tokens in preprocessed.items():
    counter = Counter(tokens)
    bow_data[doc_id] = [counter.get(term, 0) for term in vocab]

df_bow = pd.DataFrame(bow_data, index=vocab)
df_bow.index.name = 'Term'
print('\nMatriks Bag-of-Words (Raw Count):')
print(df_bow)

Ukuran vocabulary: 39 term
Term: ['ajar', 'akademik', 'akibat', 'aktif', 'beasiswa', 'buat', 'buka', 'cerdas', 'coding', 'daftar', 'digital', 'drastis', 'guna', 'guru', 'indonesia', 'informasi', 'jahat', 'jaring', 'kampus', 'kip', 'komputer', 'kuliah', 'kurikulum', 'laku', 'lalu', 'mahasiswa', 'mata', 'online', 'pelihara', 'performa', 'pilih', 'program', 'sekolah', 'serang', 'siber', 'sistem', 'terap', 'tinggi', 'tingkat']

Matriks Bag-of-Words (Raw Count):
           D1  D2  D3  D4
Term                     
ajar        0   1   0   0
akademik    1   0   0   0
akibat      0   0   0   1
aktif       0   0   1   0
beasiswa    0   0   1   0
buat        0   1   0   1
buka        0   0   1   0
cerdas      0   1   0   1
coding      0   1   0   0
daftar      0   0   1   0
digital     0   0   0   1
drastis     0   0   0   1
guna        0   0   0   1
guru        0   0   1   0
indonesia   0   1   0   1
informasi   1   0   0   0
jahat       0   0   0   1
jaring      1   0   0   0
kampus      1   0 

In [6]:
# ─── Hitung TF Manual ─────────────────────────────────────────────────────
doc_lengths = {doc_id: len(tokens) for doc_id, tokens in preprocessed.items()}

tf_data = {}
for doc_id, tokens in preprocessed.items():
    counter = Counter(tokens)
    tf_data[doc_id] = [round(counter.get(term, 0) / doc_lengths[doc_id], 4) for term in vocab]

df_tf = pd.DataFrame(tf_data, index=vocab)
df_tf.index.name = 'Term'
print('Panjang dokumen:')
for doc_id, length in doc_lengths.items():
    print(f'  {doc_id}: {length} token')
print('\nMatriks TF (Term Frequency):')
print(df_tf)

Panjang dokumen:
  D1: 10 token
  D2: 10 token
  D3: 12 token
  D4: 12 token

Matriks TF (Term Frequency):
            D1   D2      D3      D4
Term                               
ajar       0.0  0.1  0.0000  0.0000
akademik   0.1  0.0  0.0000  0.0000
akibat     0.0  0.0  0.0000  0.0833
aktif      0.0  0.0  0.0833  0.0000
beasiswa   0.0  0.0  0.0833  0.0000
buat       0.0  0.1  0.0000  0.0833
buka       0.0  0.0  0.0833  0.0000
cerdas     0.0  0.1  0.0000  0.0833
coding     0.0  0.1  0.0000  0.0000
daftar     0.0  0.0  0.0833  0.0000
digital    0.0  0.0  0.0000  0.0833
drastis    0.0  0.0  0.0000  0.0833
guna       0.0  0.0  0.0000  0.0833
guru       0.0  0.0  0.0833  0.0000
indonesia  0.0  0.1  0.0000  0.0833
informasi  0.1  0.0  0.0000  0.0000
jahat      0.0  0.0  0.0000  0.0833
jaring     0.1  0.0  0.0000  0.0000
kampus     0.1  0.0  0.0000  0.0000
kip        0.0  0.0  0.0833  0.0000
komputer   0.1  0.0  0.0000  0.0000
kuliah     0.0  0.0  0.0833  0.0000
kurikulum  0.0  0.1  0.0000  

In [7]:
# ─── Hitung DF dan IDF Manual ─────────────────────────────────────────────
N = len(documents)  # Jumlah total dokumen

df_count = {}  # document frequency per term
idf_val  = {}  # IDF per term

for term in vocab:
    # Hitung berapa dokumen yang mengandung term ini
    df_t = sum(1 for tokens in preprocessed.values() if term in tokens)
    df_count[term] = df_t
    # IDF = log10(N / df)
    idf_val[term] = round(math.log10(N / df_t), 4)

df_idf = pd.DataFrame({
    'DF (doc frequency)' : [df_count[t] for t in vocab],
    'IDF = log10(N/df)'  : [idf_val[t] for t in vocab]
}, index=vocab)
df_idf.index.name = 'Term'

print(f'Jumlah dokumen (N) = {N}')
print('\nTabel DF dan IDF:')
print(df_idf)

Jumlah dokumen (N) = 4

Tabel DF dan IDF:
           DF (doc frequency)  IDF = log10(N/df)
Term                                            
ajar                        1             0.6021
akademik                    1             0.6021
akibat                      1             0.6021
aktif                       1             0.6021
beasiswa                    1             0.6021
buat                        2             0.3010
buka                        1             0.6021
cerdas                      2             0.3010
coding                      1             0.6021
daftar                      1             0.6021
digital                     1             0.6021
drastis                     1             0.6021
guna                        1             0.6021
guru                        1             0.6021
indonesia                   2             0.3010
informasi                   1             0.6021
jahat                       1             0.6021
jaring                     

In [8]:
# ─── Hitung TF-IDF Manual ─────────────────────────────────────────────────
tfidf_manual_data = {}
for doc_id in doc_ids:
    tfidf_manual_data[doc_id] = [
        round(tf_data[doc_id][i] * idf_val[term], 4)
        for i, term in enumerate(vocab)
    ]

df_tfidf_manual = pd.DataFrame(tfidf_manual_data, index=vocab)
df_tfidf_manual.index.name = 'Term'
print('Matriks TF-IDF (Perhitungan Manual):')
print(df_tfidf_manual)

# Tampilkan term tertinggi per dokumen
print('\nTerm dengan bobot TF-IDF tertinggi per dokumen:')
for doc_id in doc_ids:
    top_term_idx = df_tfidf_manual[doc_id].idxmax()
    top_val      = df_tfidf_manual[doc_id].max()
    print(f'  {doc_id}: "{top_term_idx}" = {top_val}')

Matriks TF-IDF (Perhitungan Manual):
               D1      D2      D3      D4
Term                                     
ajar       0.0000  0.0602  0.0000  0.0000
akademik   0.0602  0.0000  0.0000  0.0000
akibat     0.0000  0.0000  0.0000  0.0502
aktif      0.0000  0.0000  0.0502  0.0000
beasiswa   0.0000  0.0000  0.0502  0.0000
buat       0.0000  0.0301  0.0000  0.0251
buka       0.0000  0.0000  0.0502  0.0000
cerdas     0.0000  0.0301  0.0000  0.0251
coding     0.0000  0.0602  0.0000  0.0000
daftar     0.0000  0.0000  0.0502  0.0000
digital    0.0000  0.0000  0.0000  0.0502
drastis    0.0000  0.0000  0.0000  0.0502
guna       0.0000  0.0000  0.0000  0.0502
guru       0.0000  0.0000  0.0502  0.0000
indonesia  0.0000  0.0301  0.0000  0.0251
informasi  0.0602  0.0000  0.0000  0.0000
jahat      0.0000  0.0000  0.0000  0.0502
jaring     0.0602  0.0000  0.0000  0.0000
kampus     0.0602  0.0000  0.0000  0.0000
kip        0.0000  0.0000  0.0502  0.0000
komputer   0.0602  0.0000  0.0000  0.00

In [ ]:
# ─── scikit-learn TF-IDF ──────────────────────────────────────────────────
corpus_for_sklearn = [' '.join(preprocessed[doc_id]) for doc_id in doc_ids]

vectorizer = TfidfVectorizer()
tfidf_sklearn_matrix = vectorizer.fit_transform(corpus_for_sklearn)

sklearn_terms  = vectorizer.get_feature_names_out()
df_tfidf_sklearn = pd.DataFrame(
    tfidf_sklearn_matrix.toarray().T,
    index=sklearn_terms,
    columns=doc_ids
).round(4)
df_tfidf_sklearn.index.name = 'Term'

print('Matriks TF-IDF (scikit-learn):')
print(df_tfidf_sklearn)

print('\nTerm dengan bobot TF-IDF tertinggi per dokumen (scikit-learn):')
for doc_id in doc_ids:
    top_term = df_tfidf_sklearn[doc_id].idxmax()
    top_val  = df_tfidf_sklearn[doc_id].max()
    print(f'  {doc_id}: "{top_term}" = {top_val}')

Matriks TF-IDF (scikit-learn):
               D1      D2      D3      D4
Term                                     
ajar       0.0000  0.3359  0.0000  0.0000
akademik   0.3289  0.0000  0.0000  0.0000
akibat     0.0000  0.0000  0.0000  0.3145
aktif      0.0000  0.0000  0.2887  0.0000
beasiswa   0.0000  0.0000  0.2887  0.0000
buat       0.0000  0.2648  0.0000  0.2480
buka       0.0000  0.0000  0.2887  0.0000
cerdas     0.0000  0.2648  0.0000  0.2480
coding     0.0000  0.3359  0.0000  0.0000
daftar     0.0000  0.0000  0.2887  0.0000
digital    0.0000  0.0000  0.0000  0.3145
drastis    0.0000  0.0000  0.0000  0.3145
guna       0.0000  0.0000  0.0000  0.3145
guru       0.0000  0.0000  0.2887  0.0000
indonesia  0.0000  0.2648  0.0000  0.2480
informasi  0.3289  0.0000  0.0000  0.0000
jahat      0.0000  0.0000  0.0000  0.3145
jaring     0.3289  0.0000  0.0000  0.0000
kampus     0.3289  0.0000  0.0000  0.0000
kip        0.0000  0.0000  0.2887  0.0000
komputer   0.3289  0.0000  0.0000  0.0000
kul

In [ ]:
# Perbandingan Nilai TF-IDF
print('Perbandingan Formula IDF:')
print(f'  Manual      : IDF(t) = log10(N / df)')
print(f'  scikit-learn: IDF(t) = log((1+N) / (1+df)) + 1, lalu dinormalisasi L2')
print()

print('Nilai IDF per term:')
idf_compare = []
for term in vocab:
    manual_idf = idf_val[term]

    if term in vectorizer.vocabulary_:
        sk_idf = round(vectorizer.idf_[vectorizer.vocabulary_[term]], 4)
    else:
        sk_idf = 'N/A'
    idf_compare.append({'Term': term, 'IDF Manual (log10)': manual_idf, 'IDF scikit-learn (smooth+1)': sk_idf})

df_compare = pd.DataFrame(idf_compare).set_index('Term')
print(df_compare)

print('\nKesimpulan:')
print('  Perbedaan nilai wajar karena perbedaan formula dan normalisasi L2.')
print('  Keduanya sepakat: term yang jarang muncul di banyak dokumen mendapat bobot lebih tinggi.')

Perbandingan Formula IDF:
  Manual      : IDF(t) = log10(N / df)
  scikit-learn: IDF(t) = log((1+N) / (1+df)) + 1, lalu dinormalisasi L2

Nilai IDF per term:
           IDF Manual (log10)  IDF scikit-learn (smooth+1)
Term                                                      
ajar                   0.6021                       1.9163
akademik               0.6021                       1.9163
akibat                 0.6021                       1.9163
aktif                  0.6021                       1.9163
beasiswa               0.6021                       1.9163
buat                   0.3010                       1.5108
buka                   0.6021                       1.9163
cerdas                 0.3010                       1.5108
coding                 0.6021                       1.9163
daftar                 0.6021                       1.9163
digital                0.6021                       1.9163
drastis                0.6021                       1.9163
guna            

## 7. Analisis Singkat

Berdasarkan perhitungan TF-IDF, term-term dengan bobot tertinggi pada setiap dokumen adalah term yang **sering muncul dalam satu dokumen tetapi jarang muncul di dokumen lain** — itulah intisari dari IDF. Misalnya, term seperti *"siber"* atau *"serang"* mendapat bobot tinggi di D4 karena hanya muncul dalam konteks keamanan siber, sementara term seperti *"indonesia"* atau *"sistem"* yang muncul di banyak dokumen justru mendapat bobot rendah meski frekuensi lokalnya tinggi. Perbedaan nilai antara perhitungan manual dan scikit-learn bukan kesalahan — keduanya menggunakan formula IDF yang berbeda: manual menggunakan `log10(N/df)`, sedangkan scikit-learn menggunakan `log((1+N)/(1+df))+1` (*smooth IDF*) untuk menghindari pembagian nol, ditambah normalisasi L2 per dokumen. Dalam konteks sistem IR nyata, scikit-learn lebih direkomendasikan karena lebih robust secara numerik.